# 1D J1-J2=0.8: LorentzRNN 

This work is part of https://arxiv.org/abs/2604.24337. In this notebook, with the training of LorentzRNN, we can see the effects of spatial clamp (value and single/double application) and learning rates on the performances of LorentzRNN. 

In [1]:
import os
import sys
sys.path.append('../utility_lorentz')
from j1j2_train_loop_lorentz import *

Hypercore Lorentzian module loaded successfully with Geoopt wrappers.
GPU Available:  False


In [2]:
E_exact = -42.07006297371643
units = 70
syssize = 100
nssamples = 80
J1 = 1.0
J2 = 0.8
lr1=5e-3
lr2=1e-3
nsteps = 1001
var_tol = 20.0
fname=f'1d_J1J2_results_Lorentz_N={syssize}_norm_spatial=18_sc=2/J2={J2}'

In [5]:
def set_cpu_deterministic(seed=111):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

set_cpu_deterministic(111)

In [4]:
def define_load_test(wf,  numsamples,path_to_weights, Ee):
    test_samples_before = wf.sample(numsamples)
    print(f'The number of samples is {len(test_samples_before)}')
    # --- PART A: Check performance BEFORE loading (Baseline) ---
    wf.model.eval() 
    with torch.no_grad():
        test_gs_before = J1J2_local_energies(wf, syssize, J1, J2, Bz, numsamples, test_samples_before, Marshall_sign=True)
        gs_mean_b = np.round(np.mean(test_gs_before),4)
        gs_var_b = np.round(np.var(test_gs_before),4)
    print(f'Before loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_b}, var E = {gs_var_b}')
    print('====================================================================')

     # --- PART B: Remap and Load the Weights ---
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    # This line loads the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        test_samples_after = wf.sample(numsamples)
        test_gs_after =  J1J2_local_energies(wf, syssize, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
        gs_mean_a = np.round(np.mean(test_gs_after),4)
        gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

## Spatial_clamp=2.0: Double clamp in forward pass

In [10]:
spatial_clamp = 2.0
fname=f'1d_J1J2_results_N=100_double_clamp/LorentzRNN_sc={spatial_clamp}/J2={J2}'

wf2=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units, spatial_clamp=spatial_clamp, seed=111)
for name, param in wf2.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")
t0=time.time()
mE, vE = run_J1J2(wf2, nsteps, syssize, var_tol, J1_=J1, J2_=J2, Marshall_sign=True, 
                   numsamples=nssamples, lr1=lr1, lr2=5e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

Layer: rnn.W | Size: torch.Size([70, 70])
Layer: rnn.U | Size: torch.Size([70, 2])
Layer: rnn.b | Size: torch.Size([70])
Layer: dense_a.weight | Size: torch.Size([2, 70])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 70])
Layer: dense_p.bias | Size: torch.Size([2])
Layer: manifold.k | Size: torch.Size([])
step: 0, loss: -1.17065, mean energy: 44.21762-0.00016j|varE: 0.34330| Hyp LR: 5.00e-03| LR: 5.00e-03| tau=1.05
max_h0 = 1.0001099109649658 | max_h_spatial_norm = 0.014826799742877483| max_h_violation = 5.960464477539063e-08
Best model saved at epoch 2 with best E=43.79929-0.03533j, varE=1.24026
Best model saved at epoch 3 with best E=43.04993+0.22856j, varE=5.28321
Best model saved at epoch 4 with best E=41.14780+0.25166j, varE=9.78447
step: 10, loss: 0.58679, mean energy: 20.31303+0.70130j|varE: 46.54021| Hyp LR: 5.00e-03| LR: 5.00e-03| tau=1.0490000000000002
max_h0 = 1.8186604976654053 | max_h_spatial_norm = 1.5190542936325073| max_h_viola

## Spatial clamp=4.0, double clamp in forward pass

In [7]:
spatial_clamp = 4.0
fname=f'1d_J1J2_results_N=100_double_clamp/LorentzRNN_sc={spatial_clamp}/J2={J2}'

wf2=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units, spatial_clamp=spatial_clamp, seed=111)
for name, param in wf2.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")
t0=time.time()
mE, vE = run_J1J2(wf2, nsteps, syssize, var_tol, J1_=J1, J2_=J2, Marshall_sign=True, 
                   numsamples=nssamples, lr1=lr1, lr2=8e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

Layer: rnn.W | Size: torch.Size([70, 70])
Layer: rnn.U | Size: torch.Size([70, 2])
Layer: rnn.b | Size: torch.Size([70])
Layer: dense_a.weight | Size: torch.Size([2, 70])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 70])
Layer: dense_p.bias | Size: torch.Size([2])
Layer: manifold.k | Size: torch.Size([])


/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1052: ComplexWarning: Casting complex values to real discards the imaginary part
  current = float(metrics)


step: 0, loss: -1.17896, mean energy: 44.22102+0.00016j|varE: 0.34093| Hyp LR: 8.00e-03| LR: 5.00e-03| tau=1.05
max_h0 = 1.0001044273376465 | max_h_spatial_norm = 0.014456120319664478| max_h_violation = 1.1920928955078125e-07
Best model saved at epoch 2 with best E=43.82293-0.00336j, varE=1.12347
Best model saved at epoch 3 with best E=43.22062-0.16930j, varE=3.06085
Best model saved at epoch 4 with best E=42.05731+0.06131j, varE=7.32197
step: 10, loss: 132.50818, mean energy: 12.06516+0.65243j|varE: 35.76869| Hyp LR: 8.00e-03| LR: 5.00e-03| tau=1.0490000000000002
max_h0 = 4.123096466064453 | max_h_spatial_norm = 3.999990701675415| max_h_violation = 5.7220458984375e-06
step: 20, loss: -128.20493, mean energy: 8.51591-0.69583j|varE: 47.89034| Hyp LR: 8.00e-03| LR: 5.00e-03| tau=1.048
max_h0 = 4.123096466064453 | max_h_spatial_norm = 3.999990701675415| max_h_violation = 5.7220458984375e-06
step: 30, loss: 77.70345, mean energy: -0.13457-0.03826j|varE: 34.27673| Hyp LR: 8.00e-03| LR: 5.00

## Spatial clamp=2.0, single clamp in forward pass

In [5]:
wf2=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units, seed=111)
for name, param in wf2.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")

Layer: rnn.W | Size: torch.Size([70, 70])
Layer: rnn.U | Size: torch.Size([70, 2])
Layer: rnn.b | Size: torch.Size([70])
Layer: dense_a.weight | Size: torch.Size([2, 70])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 70])
Layer: dense_p.bias | Size: torch.Size([2])
Layer: manifold.k | Size: torch.Size([])


### lr1=5e-3, lr2=8e-3

In [6]:
# lr2=8e-3: 
t0=time.time()
mE, vE = run_J1J2(wf2, nsteps, syssize, var_tol, J1_=J1, J2_=J2, Marshall_sign=True, 
                   numsamples=nssamples, lr1=lr1, lr2=8e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1052: ComplexWarning: Casting complex values to real discards the imaginary part
  current = float(metrics)


step: 0, loss: -1.17896, mean energy: 44.22102+0.00016j, varE: 0.34093, | Hyp LR: 8.00e-03| LR: 5.00e-03
Best model saved at epoch 2 with best E=43.82293-0.00336j, varE=1.12347
Best model saved at epoch 3 with best E=43.22062-0.16930j, varE=3.06085
Best model saved at epoch 4 with best E=42.05731+0.06131j, varE=7.32197
step: 10, loss: -25.72565, mean energy: 9.50331+0.49779j, varE: 35.46722, | Hyp LR: 8.00e-03| LR: 5.00e-03
step: 20, loss: -25.61801, mean energy: -0.91581+0.06222j, varE: 40.87911, | Hyp LR: 8.00e-03| LR: 5.00e-03
step: 30, loss: -45.36525, mean energy: -2.30441-0.31529j, varE: 31.17489, | Hyp LR: 8.00e-03| LR: 5.00e-03
Best model saved at epoch 36 with best E=-3.16977+0.02496j, varE=19.34406
step: 40, loss: -43.57817, mean energy: -3.78676-0.41998j, varE: 30.81175, | Hyp LR: 8.00e-03| LR: 5.00e-03
step: 50, loss: 15.29715, mean energy: -5.93727+0.45403j, varE: 30.80971, | Hyp LR: 8.00e-03| LR: 5.00e-03
step: 60, loss: -22.45566, mean energy: -5.68239-0.22719j, varE: 28

### lr1=5e-3, lr2=8e-3: stalled at -39

In [ ]:
#lr2=8e-3
mE, vE = run_J1J2(wf2, nsteps, syssize, var_tol, J1_=J1, J2_=J2, Marshall_sign=True, 
                   numsamples=nssamples, lr1=lr1, lr2=8e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1052: ComplexWarning: Casting complex values to real discards the imaginary part
  current = float(metrics)


step: 0, loss: -1.93115, mean energy: 44.02390-0.07420j, varE: 1.19609, | Hyp LR: 8.00e-03| LR: 5.00e-03
Best model saved at epoch 1 with best E=42.94239+0.03686j, varE=4.23232
Best model saved at epoch 2 with best E=40.25627+0.05002j, varE=10.18384
step: 10, loss: -15.61508, mean energy: 3.98070+0.23173j, varE: 37.45234, | Hyp LR: 8.00e-03| LR: 5.00e-03
step: 20, loss: 0.14781, mean energy: -1.24725+0.09452j, varE: 30.18226, | Hyp LR: 8.00e-03| LR: 5.00e-03
step: 30, loss: 45.19559, mean energy: -2.42966-0.14030j, varE: 32.46266, | Hyp LR: 8.00e-03| LR: 5.00e-03
step: 40, loss: 23.89987, mean energy: -6.42341-0.24134j, varE: 29.55037, | Hyp LR: 8.00e-03| LR: 5.00e-03
Best model saved at epoch 43 with best E=-7.06271-0.46036j, varE=19.86843
step: 50, loss: 1.63821, mean energy: -7.19239+0.58605j, varE: 21.74952, | Hyp LR: 8.00e-03| LR: 5.00e-03
Best model saved at epoch 53 with best E=-9.60186-0.00223j, varE=19.86457
step: 60, loss: 10.41968, mean energy: -9.94435-0.42885j, varE: 27.75

### lr1=5e-3, lr2=1e-3:stalled at -37

In [ ]:
# lr2=1e-3: stuck at -37.
#step: 40, loss: -10.02596, mean energy: 3.56952-0.43348j, varE: 24.61323, current_lr = [0.005]
mE, vE = run_J1J2(wf2, nsteps, syssize, var_tol, J1_=J1, J2_=J2, Marshall_sign=True, 
                   numsamples=nssamples, lr1=lr1, lr2=lr2, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1052: ComplexWarning: Casting complex values to real discards the imaginary part
  current = float(metrics)


step: 0, loss: -1.93115, mean energy: 44.02390-0.07420j, varE: 1.19609, | Hyp LR: 1.00e-03| LR: 5.00e-03
Best model saved at epoch 1 with best E=42.96088+0.08175j, varE=3.78738
Best model saved at epoch 2 with best E=40.49235+0.03450j, varE=9.42988
Best model saved at epoch 3 with best E=36.64846+0.39279j, varE=18.42280
step: 10, loss: 0.14252, mean energy: 5.86467+0.24307j, varE: 32.92953, | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 20, loss: -4.97162, mean energy: -2.17896-0.49474j, varE: 30.53804, | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 30, loss: 15.68831, mean energy: -3.94550+0.22240j, varE: 29.88675, | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 40, loss: 3.80521, mean energy: -4.48467-0.80420j, varE: 38.50511, | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 50, loss: -5.46506, mean energy: -6.60374-0.12178j, varE: 26.52761, | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 60, loss: 0.41693, mean energy: -9.50673-0.05289j, varE: 37.61443, | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 70, loss: 3.54230, mean energy: -1